# Hierarchical Data

## JSON

JavaScript Object Notation, or **JSON**, is a popular data format for representing hierarchical data. Despite its name, its application extends far beyond JavaScript, the language for which it was originally designed.

Let's read in a JSON file and print out the first observation. (_Warning:_ Never try to print the entire contents of a JSON file in a notebook; this will freeze the notebook if the file is large!)

In [ ]:
import requests

# Fetch data from a URL
response = requests.get("https://dlsun.github.io/pods/data/tvshows.json")

# JSON object can be accessed directly from the response.
data_shows = response.json()

data_shows[0]

Now let's investigate the JSON data that we just loaded, again being careful not to print out all of the data. Let's start by looking at the top-level variables associated with each TV show.

In [ ]:
show = data_shows[0]  # data for the first TV show
show.keys()

We see variables like **`name`** and **`network`**, but also "variables" like **`cast`** and **`seasons`**, which contain multiple values.

In [ ]:
show["name"]

In [ ]:
show["cast"]

A "variable" (like **`cast`**) with multiple values is called a _repeated field_. A repeated field might itself contain a repeated field (e.g., each show has multiple seasons, and each season in turn has multiple episodes), creating a hierarchy of variables. Repeated fields are represented as lists or arrays in JSON.

Let's take a closer look at how each cast member is represented, by zooming in on the first cast member.

In [ ]:
show["cast"][0]

It appears that each cast member is itself a dictionary with four keys: **`person`** (i.e., the actor), **`character`**, **`self`**, and **`voice`**. The first two attributes are themselves dictionaries containing further information about the actor and the character, while the last two attributes are booleans.

If we wanted to know which show had the biggest cast, we could first get the show name and actor name by traversing the levels using nested loops:

In [ ]:
shows = []
actors = []
for show in data_shows:
  for castmember in show["cast"]:
    # exclude voice actors
    if not castmember["voice"]:
      shows.append(show["name"])
      actors.append(castmember["person"]["name"])

shows, actors

However, it is often easier to work with hierarchical data by first flattening it to a `DataFrame`.

### Flattening Hierarchical Data

Although hierarchical data cannot be efficiently represented using a `DataFrame`, most questions do not require all of the information in the data. In these cases, it is helpful to first "flatten" the hierarchical data into a `DataFrame`.

For example, suppose we want to know the average runtime of shows. To answer this question, it suffices to work with a `DataFrame` with one row per show. We can use the `json_normalize()` function in `pandas` to flatten the data into a `DataFrame` of this form.

In [ ]:
import pandas as pd

df_shows = pd.json_normalize(data_shows)


In [ ]:
df_shows["runtime"].mean()

Let us take a closer look at the columns of this `DataFrame`.

In [ ]:
df_shows.columns

Notice that:

- Fields that were themselves dictionaries, such as **`network`**, have been expanded into multiple columns, with names like **`network.name`**, **`network.country.name`**, etc.
- Repeated fields, such as **`cast`** and **`seasons`**, are just columns in this `DataFrame`.

Let's take a look at one of these repeated fields.

In [ ]:
df_shows["cast"]

These columns just contain a dump of the raw JSON. The information in these columns is not readily accessible.

### A Simple Example

Now, suppose we want to get a list of the non-voice actors in the data set, like we did above, but using `pd.json_normalize()`?

We can specify the observational unit in `pd.json_normalize()`. So if we wanted a `DataFrame` where each row represents a cast member, we would pass in the name of that variable in the JSON data (i.e., **`cast`**) to `json_normalize()`.

In [ ]:
df_actors = pd.json_normalize(data_shows, "cast")
df_actors

There is just one problem. We have lost all the fields in the data in the levels _above_ **`cast`**. If we want to keep any fields from these levels, we must specify them explicitly in the `meta=` argument.

For example, we might want to know the **`name`** of the TV show. So we specify `meta="name"` (as well as a `meta_prefix` to avoid naming conflicts).

In [ ]:
df_actors = pd.json_normalize(data_shows, "cast",
                              meta="name", meta_prefix="tvshow.")
df_actors

Now it's easy to determine the show with the biggest cast.

In [ ]:
df_actors[~df_actors["voice"]]["tvshow.name"].value_counts()

## Exercise

- What shows is Lena Dunham in?

- Who appears in the most (unique) TV shows?

## XML

First, we read in the data using a library called BeautifulSoup.

In [ ]:
from bs4 import BeautifulSoup
response = requests.get("https://dlsun.github.io/pods/data/tvshows.xml")
soup = BeautifulSoup(response.text, 'xml')

To determine which show had the biggest cast, we can:

- search for a tag using `.find_all()` or `.find()` (which returns the first tag found)
- navigate the tree using `.parent` and `.children`.

In [ ]:
actors = []
show_names = []
for actor in soup.find_all("person"):
  if actor.parent.find("voice").string == "false":
    actors.append(actor.find("name").string)
    show_names.append(actor.parent.parent.find("name").string)

actors, show_names

If we want to further process this data, we have to make a `DataFrame` ourselves. There is no analog of `pd.json_normalize()` for XML.

In [ ]:
df_actors_xml = pd.DataFrame({
    "tvshow": show_names,
    "actor": actors
})


## Exercise

- What shows is Lena Dunham in?

- Who appears in the most (unique) TV shows?